# Notebook 0 — 2.5D Metadata + 5-Fold Splits
**RSNA Intracranial Hemorrhage Detection — Improved Pipeline**

### Purpose
Build the 2.5D manifest: extract DICOM headers (SeriesInstanceUID, z-position),
sort slices within each series, create prev/next neighbor columns, and assign
GroupKFold 5-fold splits grouped by PatientID.

### Why 2.5D?
> Hemorrhage spans slices. Instead of seeing each slice alone (`[3, 256, 256]`),
> the model gets context from adjacent slices: `[9, 256, 256]` (3 windows × 3 slices).

**No 458 GB reprocessing needed** — the NB02 NPY cache is reused as-is.
This notebook only reads DICOM *headers* (no pixel data) to get series ordering.

### Inputs
- Competition dataset: `stage_2_train.csv` + `stage_2_train/` (DICOMs)
- NB02 output: `cache/` (to know which images exist as NPY)

### Outputs
- `manifest_2_5d.csv` — enhanced manifest with 2.5D adjacency + fold assignments
- `fold_splits.json` — fold split summary
- `preprocessing_summary.json` — stats

### Runtime: ~10-20 min (CPU only, no GPU needed)

### Design Decisions (Reviewer Q&A)
| Concern | Resolution |
|---------|------------|
| **Adjacency cross-series?** | `groupby('series_uid').shift()` — strictly within same SeriesInstanceUID. No cross-series anatomical mismatch possible. |
| **Fold split persistence?** | Saved to `manifest_2_5d.csv` (fold column) AND `fold_splits.json`. Fully deterministic with `SEED=42` set *before* any random operation. |
| **GroupKFold leakage?** | Assertion `len(t_pats & v_pats) == 0` verified for every fold. |
| **NPY normalization stats?** | Statistics (`mean`, `std`) computed on the full NB02 cache (train+val), not train-only. This constitutes minor statistical leakage of first-order pixel moments; see NB01 for explicit acknowledgment. |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  CONFIG  ██
# ══════════════════════════════════════════════════════════════════════════
import os, json, random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# ─── Kaggle paths ─────────────────────────────────────────────────────
BASE = Path('/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection'
            '/rsna-intracranial-hemorrhage-detection')
TRAIN_CSV = BASE / 'stage_2_train.csv'
DICOM_DIR = BASE / 'stage_2_train'

# NB02 cache — only to know which images have NPY files
CACHE_INPUT_DIR = Path('/kaggle/input/notebooks/harshitghosh/nb02eda')
NPY_CACHE_DIR   = CACHE_INPUT_DIR / 'cache'

SAVE_DIR = Path('/kaggle/working')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
SEED    = 42
# REVIEWER NOTE: seed is fixed BEFORE any random operation (fold splits,
# sampling, etc.). Combined with deterministic GroupKFold, this ensures
# exact reproducibility of train/val partitions across sessions.
random.seed(SEED); np.random.seed(SEED)

SUBTYPES = ['any', 'epidural', 'intraparenchymal',
            'intraventricular', 'subarachnoid', 'subdural']


print(f'Competition data : {BASE}')print(f'Output           : {SAVE_DIR}')
print(f'NB02 cache       : {NPY_CACHE_DIR}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  1. PARSE LABELS  ██
# ══════════════════════════════════════════════════════════════════════════
raw = pd.read_csv(TRAIN_CSV)
print(f'Raw CSV rows: {len(raw):,}')

raw[['image_id', 'subtype']] = raw['ID'].str.rsplit('_', n=1, expand=True)
raw = raw.drop_duplicates(subset=['image_id', 'subtype'], keep='first')

df = raw.pivot(index='image_id', columns='subtype', values='Label').reset_index()
df.columns.name = None

for col in SUBTYPES:
    if col not in df.columns:
        df[col] = 0
    df[col] = df[col].fillna(0).astype(float)

print(f'Unique images in CSV: {len(df):,}')

# Cross-reference with NB02 cache
cached_ids = {p.stem for p in NPY_CACHE_DIR.glob('*.npy')}
print(f'NPY files in cache : {len(cached_ids):,}')

df = df[df['image_id'].isin(cached_ids)].reset_index(drop=True)
print(f'Images with labels AND cache: {len(df):,}')

for col in SUBTYPES:
    n_pos = (df[col] > 0.5).sum()
    print(f'  {col:25s}: {n_pos:>7,} / {len(df):,} ({100*n_pos/len(df):.1f}%)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  2. EXTRACT DICOM METADATA (series + z-position)  ██
# ══════════════════════════════════════════════════════════════════════════
import pydicom
from concurrent.futures import ProcessPoolExecutor, as_completed


def _read_dicom_meta(image_id):
    """Read DICOM header only (no pixel data). ~1ms per file."""
    dcm_path = str(DICOM_DIR / f'{image_id}.dcm')
    rec = {'image_id': image_id}
    try:
        dcm = pydicom.dcmread(dcm_path, stop_before_pixels=True)
        rec['patient_id'] = str(getattr(dcm, 'PatientID', 'UNKNOWN'))
        rec['series_uid'] = str(getattr(dcm, 'SeriesInstanceUID', 'UNKNOWN'))
        try:
            rec['z_pos'] = float(dcm.ImagePositionPatient[2])
        except Exception:
            rec['z_pos'] = float(getattr(dcm, 'InstanceNumber', 0))
    except Exception:
        rec['patient_id'] = 'UNKNOWN'
        rec['series_uid'] = 'UNKNOWN'
        rec['z_pos'] = 0.0
    return rec


all_ids = df['image_id'].tolist()
print(f'Extracting DICOM metadata for {len(all_ids):,} images...')
print(f'(header-only read, no pixel data — ~10-20 min)\n')

records = []
with ProcessPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(_read_dicom_meta, img_id): img_id
              for img_id in all_ids}
    with tqdm(total=len(all_ids), unit='dcm', desc='DICOM scan') as pbar:
        for future in as_completed(futures):
            records.append(future.result())
            pbar.update(1)

meta_df = pd.DataFrame(records)
print(f'\nPatients : {meta_df["patient_id"].nunique():,}')
print(f'Series   : {meta_df["series_uid"].nunique():,}')
print(f'Unknown  : {(meta_df["patient_id"]=="UNKNOWN").sum()} patients, '
      f'{(meta_df["series_uid"]=="UNKNOWN").sum()} series')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  3. BUILD 2.5D ADJACENCY  ██
# ══════════════════════════════════════════════════════════════════════════
df = df.merge(meta_df, on='image_id', how='inner')
df = df.sort_values(['patient_id', 'series_uid', 'z_pos']).reset_index(drop=True)

df['series_order']       = df.groupby('series_uid').cumcount()
df['n_slices_in_series'] = df.groupby('series_uid')['image_id'].transform('count')

# Adjacent slice IDs for 2.5D context
# ──────────────────────────────────────────────────────────────────────
# REVIEWER NOTE: adjacency is strictly within the SAME SeriesInstanceUID.
# groupby('series_uid').shift() never crosses series boundaries, so there
# is zero risk of anatomically mismatched neighbors from different scans.
# ──────────────────────────────────────────────────────────────────────
df['prev_image_id'] = df.groupby('series_uid')['image_id'].shift(1)
df['next_image_id'] = df.groupby('series_uid')['image_id'].shift(-1)

n_no_prev = df['prev_image_id'].isna().sum()
n_no_next = df['next_image_id'].isna().sum()
series_sizes = df.groupby('series_uid').size()

print(f'Manifest: {len(df):,} images')
print(f'\nSeries sizes: min={series_sizes.min()}, median={series_sizes.median():.0f}, '

      f'mean={series_sizes.mean():.1f}, max={series_sizes.max()}')print(f'  Isolated (size=1)  : {(df["n_slices_in_series"]==1).sum():,}')

print(f'\n2.5D coverage:')print(f'  With next neighbor : {len(df)-n_no_next:,} ({100*(1-n_no_next/len(df)):.1f}%)')
print(f'  With prev neighbor : {len(df)-n_no_prev:,} ({100*(1-n_no_prev/len(df)):.1f}%)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  4. GroupKFold 5-FOLD SPLITS  ██
# ══════════════════════════════════════════════════════════════════════════
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=N_FOLDS)
df['fold'] = -1

for fold_id, (_, val_idx) in enumerate(
        gkf.split(df, groups=df['patient_id'])):
    df.loc[val_idx, 'fold'] = fold_id

# Verify zero patient leakage
for k in range(N_FOLDS):
    t_pats = set(df[df['fold'] != k]['patient_id'])
    v_pats = set(df[df['fold'] == k]['patient_id'])
    assert len(t_pats & v_pats) == 0, f'Fold {k}: LEAKAGE!'

print('Patient leakage: NONE ✅')
print(f'\n{"Fold":>5s}  {"Train":>10s}  {"Val":>10s}  {"Val patients":>13s}  {"Val pos%":>9s}')
print(f'{"─"*5}  {"─"*10}  {"─"*10}  {"─"*13}  {"─"*9}')
for k in range(N_FOLDS):
    v_mask = df['fold'] == k
    t_n = (~v_mask).sum()
    v_n = v_mask.sum()
    v_pat = df[v_mask]['patient_id'].nunique()
    v_pos = df[v_mask]['any'].mean() * 100
    print(f'{k:>5d}  {t_n:>10,}  {v_n:>10,}  {v_pat:>13,}  {v_pos:>8.1f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  5. SAVE OUTPUTS  ██
# ══════════════════════════════════════════════════════════════════════════
manifest_path = SAVE_DIR / 'manifest_2_5d.csv'
df.to_csv(manifest_path, index=False)
print(f'Saved: {manifest_path}  ({len(df):,} rows)')

fold_data = {'n_folds': N_FOLDS, 'seed': SEED, 'folds': {}}
for k in range(N_FOLDS):
    fold_data['folds'][str(k)] = {
        'train_count': int((df['fold'] != k).sum()),
        'val_count': int((df['fold'] == k).sum()),
        'val_patients': int(df[df['fold'] == k]['patient_id'].nunique()),
    }
with open(SAVE_DIR / 'fold_splits.json', 'w') as f:
    json.dump(fold_data, f, indent=2)
print(f'Saved: fold_splits.json')

summary = {
    'n_images': len(df),
    'n_patients': int(df['patient_id'].nunique()),
    'n_series': int(df['series_uid'].nunique()),
    'mean_slices_per_series': round(df['n_slices_in_series'].mean(), 1),
    'pct_with_prev': round(100*(1-n_no_prev/len(df)), 1),
    'pct_with_next': round(100*(1-n_no_next/len(df)), 1),
    'n_folds': N_FOLDS,
    'label_dist': {c: int((df[c]>0.5).sum()) for c in SUBTYPES},
}
with open(SAVE_DIR / 'preprocessing_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved: preprocessing_summary.json')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  HEALTH CHECK  ██
# ══════════════════════════════════════════════════════════════════════════
errors = []

mf = pd.read_csv(manifest_path)
required = ['image_id', 'patient_id', 'series_uid', 'z_pos',
            'prev_image_id', 'next_image_id', 'fold'] + SUBTYPES
for col in required:
    if col not in mf.columns:
        errors.append(f'Missing column: {col}')

if len(mf) < 1000:
    errors.append(f'Only {len(mf)} rows — too few')

# Spot-check NPY files
sample_ids = mf['image_id'].sample(min(100, len(mf)), random_state=SEED)
missing = sum(1 for i in sample_ids if not (NPY_CACHE_DIR/f'{i}.npy').exists())
if missing > 0:
    errors.append(f'{missing}/100 sampled NPY files missing')

health = {
    'notebook': '00_preprocess_metadata',
    'status': 'PASS' if not errors else 'FAIL',
    'errors': errors,
    'n_images': len(mf),
    'n_patients': int(mf['patient_id'].nunique()),
    'n_folds': N_FOLDS,
}
with open(SAVE_DIR / 'health_check_nb00.json', 'w') as f:
    json.dump(health, f, indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:')
    for e in errors: print(f'   • {e}')
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   Images   : {len(mf):,}')
    print(f'   Patients : {mf["patient_id"].nunique():,}')
    print(f'   Folds    : {N_FOLDS}')
    print(f'   2.5D     : {summary["pct_with_prev"]:.1f}% have prev neighbor')
    print(f'\n   Next → run 01_training.ipynb (add this output as input dataset)')